
LPR Thorium Analysis: Data vs. MC Selection Efficiencies

Purpose:
1. Load summary files for processed Thorium data and Monte Carlo (MC).
2. Calculate the stepwise and cumulative selection efficiencies for both.
3. Compute binomial errors for all efficiencies.
4. Calculate the Data/MC efficiency ratio (scale factor) at each step.
5. Propagate errors to determine the uncertainty on the ratio.

In [1]:
import sys
sys.path.append('/lhome/ific/c/ccortesp/Analysis/')

from libs import crudo

import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import yaml

# Styling Plot
crudo.pt.ccortesp_plot_style()
# PRELIM_LOGO = np.asarray(Image.open('/lhome/ific/c/ccortesp/Analysis/images/next_logo_preliminary.png'))

%matplotlib inline
%load_ext autoreload
%autoreload 2


In [2]:
# -------------------
# LOAD CONFIGURATIONS
# -------------------
CONFIG_FILE = '/lhome/ific/c/ccortesp/Analysis/NEXT-100/Th_analysis/txt/configs/he_analysis_config.yaml'

try:
    with open(CONFIG_FILE, 'r') as f:
        all_configs = yaml.safe_load(f)
except FileNotFoundError:
    raise FileNotFoundError(f"Configuration file '{CONFIG_FILE}' not found. Please check the path and try again.")

# Set up the configuration
common_config = all_configs['common']


DATA_SUMMARY_FILE =  os.path.join(common_config['summary_dir'], 'summary_LPR_p2_v2_updated.csv')
MC_SUMMARY_FILE = os.path.join(common_config['summary_dir'], 'summary_calibration_lpr_updated.csv')

In [7]:
# ===================================================================
# --- 2. DATA LOADING AND PREPARATION ---
# ===================================================================

# Load the summary files
df_data = pd.read_csv(DATA_SUMMARY_FILE).drop(columns=['Unnamed: 0'], errors='ignore')  # Drop the index column if it exists
df_mc = pd.read_csv(MC_SUMMARY_FILE).drop(columns=['Unnamed: 0'], errors='ignore')  # Drop the index column if it exists

# df_data

# --- Standardize column names for consistent processing ---
# This is a crucial step to make the code reusable.
df_data.rename(columns={'Run_ID': 'ID', 'OK': 'Triggered', 'Z_Positive': 'Extra'}, inplace=True)
df_mc.rename(columns={'Isotope': 'ID', 'Saved': 'Triggered', 'Strong_S2': 'Extra'}, inplace=True)

# df_data

print("--- Data Summary (first 5 rows) ---")
print(df_data.head())
print("\n--- MC Summary (first 5 rows) ---")
print(df_mc.head())

# --- Define the cutflow stages ---
# The list defines the sequence of cuts. Each entry is (column_before_cut, column_after_cut)
CUTFLOW_STAGES = [
                    ('Triggered', 'Sophronia'),
                    ('Sophronia', 'Clean'),
                    ('Clean', 'Extra'),
                    ('Extra', 'S1_Cut'),
                    ('S1_Cut', 'Reconstructed'),
                    ('Triggered', 'Reconstructed'),
                    ('Reconstructed', 'Inclusive'),
                    ('Inclusive', 'Fiducial')
                ]

--- Data Summary (first 5 rows) ---
      ID  Duration  Triggered    LOST  Sophronia    Clean    Extra  S1_Cut  \
0  15589     86440    1297585  982208    1296989  1296985  1006331  792282   

   Reconstructed  Inclusive  Fiducial  Single_Track  
0         614188     254528    125613         90864  

--- MC Summary (first 5 rows) ---
      ID  Generated  Interacting  Triggered  Sophronia   Clean   Extra  \
0  Tl208    5000000       901498     132594     131397  131384  131384   

   S1_Cut  Reconstructed  Inclusive  Fiducial  Single_Track  
0  131001          98496      61764     27283         21335  


In [8]:
# ===================================================================
# --- 4. CORE CALCULATION LOOP ---
# ===================================================================

results = []

print("\n--- Calculating Efficiencies and Ratios ---\n")

for stage_total, stage_pass in CUTFLOW_STAGES:

    if stage_total == '' and stage_pass == '':
        results.append()
    
    # --- DATA ---
    N_pass_data = df_data[stage_pass].sum()
    N_lost_data = df_data[stage_total].sum() - N_pass_data
    eff_data, err_data = crudo.ff.efficiency(N_pass_data, N_lost_data)
    
    # --- MC ---
    N_pass_mc = df_mc[stage_pass].sum()
    N_lost_mc = df_mc[stage_total].sum() - N_pass_mc
    eff_mc, err_mc = crudo.ff.efficiency(N_pass_mc, N_lost_mc)
    
    # --- DATA/MC RATIO (Scale Factor) ---
    if eff_mc == 0:
        ratio = 0.0
        err_ratio = 0.0
    else:
        ratio = eff_data / eff_mc
        # Error propagation for division: R = A/B -> err_R = R * sqrt((err_A/A)^2 + (err_B/B)^2)
        err_ratio = ratio * np.sqrt((err_data / eff_data)**2 + (err_mc / eff_mc)**2) if eff_data > 0 else 0.0

    # Store results
    results.append({
        'Step': stage_pass,
        'Eff_Data': eff_data,
        'Err_Data': err_data,
        'Eff_MC': eff_mc,
        'Err_MC': err_mc,
        'Ratio_Data_MC': ratio,
        'Err_Ratio': err_ratio
    })

# Convert results to a pandas DataFrame for nice display
df_results = pd.DataFrame(results)


--- Calculating Efficiencies and Ratios ---



In [9]:
# ===================================================================
# --- 5. RESULTS PRESENTATION ---
# ===================================================================

print("="*60)
print("          Final Efficiency and Data/MC Ratio Results")
print("="*60)

# Create nicely formatted strings with errors
df_results['Data Efficiency'] = df_results.apply(lambda row: f"{row.Eff_Data:.4f} ± {row.Err_Data:.4f}", axis=1)
df_results['MC Efficiency'] = df_results.apply(lambda row: f"{row.Eff_MC:.4f} ± {row.Err_MC:.4f}", axis=1)
df_results['Data/MC Ratio'] = df_results.apply(lambda row: f"{row.Ratio_Data_MC:.4f} ± {row.Err_Ratio:.4f}", axis=1)

# Display the final, user-friendly table
display(df_results[['Step', 'Data Efficiency', 'MC Efficiency', 'Data/MC Ratio']])

          Final Efficiency and Data/MC Ratio Results


,Step,Data Efficiency,MC Efficiency,Data/MC Ratio
0,Sophronia,0.9995 ± 0.0000,0.9910 ± 0.0003,1.0086 ± 0.0003
1,Clean,1.0000 ± 0.0000,0.9999 ± 0.0000,1.0001 ± 0.0000
2,Extra,0.7759 ± 0.0004,1.0000 ± 0.0000,0.7759 ± 0.0004
3,S1_Cut,0.7873 ± 0.0004,0.9971 ± 0.0001,0.7896 ± 0.0004
4,Reconstructed,0.7752 ± 0.0005,0.7519 ± 0.0012,1.0310 ± 0.0018
5,Reconstructed,0.4733 ± 0.0004,0.7428 ± 0.0012,0.6372 ± 0.0012
6,Inclusive,0.4144 ± 0.0006,0.6271 ± 0.0015,0.6609 ± 0.0019
7,Fiducial,0.4935 ± 0.0010,0.4417 ± 0.0020,1.1172 ± 0.0055
